In [297]:
%reset -f

In [298]:
# Hyper parameters
batch_size = 64
learning_rate = 2e-4
epochs = 200

In [299]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import numpy
import scipy.stats
import os
import pandas as pd
from typing import Callable

In [300]:
# device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
device = "cpu"

In [301]:
def list_from_str(str_list: str, fn_apply_to_items: Callable) -> list:
    if str_list in ["", "[]"]:
        return []

    elems = str_list.strip("[]\"").split(", ")
    return list(map(fn_apply_to_items, elems))

In [302]:
def normalize_mean_list(means_list: list[float]) -> dict[str, int | float]:
    means = torch.tensor(means_list, dtype=torch.float)

    assert means.dim() == 1

    if means.numel() == 0:
        return {
            "mean": 0,
            "median": 0,
            "range": 0,
            "std": 0,
            "var": 0,
            "medianAd": 0,
            "meanAd": 0,
        }

    mean = means.mean(0)
    median = means.median()
    range = means.max().item() - means.min().item()

    std = 0 if means.numel() == 1 else means.std(0).item()
    var = std ** 2
    median_ad = float(scipy.stats.median_abs_deviation(means.numpy()))
    mean_ad = means.sub(mean).absolute().mean()

    assert isinstance(median, torch.Tensor)

    return {
        "mean": mean.item(),
        "median": median.item(),
        "range": range,
        "std": std,
        "var": var,
        "medianAd": median_ad,
        "meanAd": mean_ad.item(),
    }

normalize_mean_list([2, 2, 3, 4, 14])

{'mean': 5.0,
 'median': 3.0,
 'range': 12.0,
 'std': 5.099019527435303,
 'var': 26.000000141166538,
 'medianAd': 1.0,
 'meanAd': 3.5999999046325684}

In [303]:
class PIIStateDataset(Dataset):
    def __init__(self, data_file: str) -> None:
        df = pd.read_csv(data_file)

        self.numUnstable = torch.tensor(df["numUnstable"], dtype=torch.float)
        self.numNM1 = torch.tensor(df["numNM1"], dtype=torch.float)
        self.numNM2 = torch.tensor(df["numNM2"], dtype=torch.float)

        # TODO: DEAL WITH MEANS
        
        start, end = df.columns.slice_locs("matchingAM","nm2RAM")
        mean_columns = df.iloc[:, start:end]
        self.meanData = {}

        for col in mean_columns:
            meanDicts = [normalize_mean_list(list_from_str(means, float)) for means in df[col]]

            for key in meanDicts[0].keys():
                valueList = [meanDict[key] for meanDict in meanDicts]
                self.meanData[col + key.title()] = torch.tensor(valueList, dtype=torch.float)

        print(self.meanData)

        self.numEdges = torch.tensor(df["numEdges"], dtype=torch.float)
        self.numSingletons = torch.tensor(df["numSingletons"], dtype=torch.float)
        self.numChains = torch.tensor(df["numChains"], dtype=torch.float)
        self.numCycles = torch.tensor(df["numCycles"], dtype=torch.float)

        self.avgChainLen = torch.tensor(df["avgChainLen"], dtype=torch.float)
        self.avgCycleLen = torch.tensor(df["avgCycleLen"], dtype=torch.float)
        
        self.converges = torch.tensor(df["converges"], dtype=torch.long)
        self.convergesOneHot = nn.functional.one_hot(self.converges, 2).float()

    def __len__(self):
        return len(self.converges)

    def __getitem__(self, idx):
        singleton_features = torch.tensor([
            self.numUnstable[idx], self.numNM1[idx], self.numNM2[idx],
            self.numEdges[idx], self.numSingletons[idx], self.numChains[idx], self.numCycles[idx],
            self.avgChainLen[idx], self.avgCycleLen[idx]
        ])
        means_stack = torch.stack(tuple(self.meanData.values()), dim=0)
        X = torch.cat((singleton_features, means_stack[:, idx]), dim=0)

        return X, self.convergesOneHot[idx]

In [304]:
training_data = PIIStateDataset("stateData_2000_10.csv")
test_data = PIIStateDataset("stateData_2000_10_test.csv")

train_dataloader = DataLoader(training_data, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

{'matchingAMMean': tensor([3.8500, 4.9500, 4.0000,  ..., 4.8500, 3.9500, 3.9000]), 'matchingAMMedian': tensor([3.0000, 4.5000, 3.5000,  ..., 4.5000, 4.0000, 3.5000]), 'matchingAMRange': tensor([5.5000, 4.0000, 6.5000,  ..., 5.5000, 6.5000, 7.5000]), 'matchingAMStd': tensor([1.9727, 1.3218, 1.8559,  ..., 1.4916, 1.9214, 2.3190]), 'matchingAMVar': tensor([3.8917, 1.7472, 3.4444,  ..., 2.2250, 3.6917, 5.3778]), 'matchingAMMedianad': tensor([1.7500, 0.7500, 0.7500,  ..., 0.5000, 1.2500, 1.0000]), 'matchingAMMeanad': tensor([1.7500, 1.0400, 1.3000,  ..., 1.0200, 1.3600, 1.6800]), 'matchingGMMean': tensor([3.2600, 4.0169, 2.7185,  ..., 3.5455, 3.7321, 2.4454]), 'matchingGMMedian': tensor([3.0000, 3.4641, 2.0000,  ..., 2.8284, 3.4641, 1.0000]), 'matchingGMRange': tensor([6.3246, 7.3485, 8.0000,  ..., 8.0000, 6.4833, 8.4853]), 'matchingGMStd': tensor([2.3973, 2.0987, 2.6055,  ..., 2.5451, 1.9640, 3.0768]), 'matchingGMVar': tensor([5.7469, 4.4047, 6.7888,  ..., 6.4773, 3.8574, 9.4666]), 'matchi

In [305]:
training_data[0][0].shape

torch.Size([107])

In [306]:
for i in range(10):
    print(i)
    print(training_data[i])

0
(tensor([17.0000,  4.0000,  2.0000,  2.0000,  1.0000,  1.0000,  0.0000,  2.0000,
         0.0000,  3.8500,  3.0000,  5.5000,  1.9727,  3.8917,  1.7500,  1.7500,
         3.2600,  3.0000,  6.3246,  2.3973,  5.7469,  1.9457,  2.0600,  2.1765,
         1.5000,  5.5000,  1.5607,  2.4357,  1.0000,  1.2664,  1.6048,  0.0000,
         5.4772,  1.9400,  3.7636,  0.0000,  1.6992, -3.1471, -3.0000,  5.0000,
         1.4765,  2.1801,  1.0000,  1.1315,  1.3750,  1.5000,  2.5000,  1.0308,
         1.0625,  0.5000,  0.6875,  0.6124,  0.0000,  2.4495,  1.2247,  1.5000,
         0.0000,  0.9186, -3.3750, -3.0000,  3.5000,  1.4930,  2.2292,  0.5000,
         1.0625,  4.8750,  4.0000,  6.0000,  2.5290,  6.3958,  1.7500,  1.8750,
         3.7866,  1.7321,  7.9373,  3.5891, 12.8819,  2.7386,  2.9206,  0.2500,
         0.0000,  5.5000,  2.3979,  5.7500,  1.2500,  1.7500,  6.5000,  5.0000,
         3.0000,  2.1213,  4.5000,  1.5000,  1.5000,  6.4181,  4.8990,  3.0383,
         2.1484,  4.6156,  1.5191,  1

In [307]:
# # Check overlap between training and testing
# df_train = pd.read_csv(f"sign_{n}_{perm_total}.csv")
# df_test = pd.read_csv(f"sign_{n}_{perm_total}_test.csv")

# set(df_train["permutation"]).intersection(set(df_test["permutation"]))

In [308]:
# Define model
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()

        self.linear_relu_stack = nn.Sequential(
            nn.Linear(training_data[0][0].size(0), 16),
            nn.ReLU(),
            nn.Linear(16, 16),
            nn.ReLU(),
            nn.Linear(16, 2),
            # nn.Softmax()
            # nn.ReLU()
        )

    def forward(self, x):
        # x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [309]:
model = NeuralNetwork().to(device)

In [310]:
# loss_fn = nn.CrossEntropyLoss()
# loss_fn = nn.BCELoss()
loss_fn = nn.BCEWithLogitsLoss()

# optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
# optimizer = torch.optim.ASGD(model.parameters(), lr=learning_rate)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
# optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

In [311]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    train_loss, correct = 0, 0

    # Set the model to training mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        # Compute prediction and loss
        pred = model(X)

        # if batch == 0:
        #     print(pred)
        #     print(y, y.shape)
        #     print(pred.argmax(1).eq(y.argmax(1)))
        #     print(pred.argmax(1).eq(y.argmax(1)).sum().item())

        loss = loss_fn(pred, y)
        train_loss += loss.item()

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

        correct += pred.argmax(1).eq(y.argmax(1)).sum().item()

    correct /= size
    train_loss /= num_batches

    # print(f"Train Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {train_loss:>8f} \n")

    return train_loss, correct


def test_loop(dataloader, model, loss_fn):
    # Set the model to evaluation mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # NOTE: This is for sanity checking the argmax tensor accuracy arithmetic
    # sanity_correct = 0
    # sanity_num_tests = 0

    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for batch, (X, y) in enumerate(dataloader):
            assert isinstance(y, torch.Tensor)

            pred: torch.Tensor = model(X)
            test_loss += loss_fn(pred, y).item()

            # if batch < 5:
            #     print(pred)
            #     print(y)
            #     print("---")
            #     print(pred.argmax(1))
            #     print(y.argmax(1))
            #     print("---")
            #     print(pred.argmax(1).eq(y.argmax(1)).sum().item())

            # NOTE: Part of the sanity checking accuracy
            # for row, label in zip(pred, y):
            #     # print(row, label)
            #     # print(row.argmax(0), label.argmax(0))
            #     if row.argmax(0).item() == label.argmax(0).item():
            #         sanity_correct += 1
            #     sanity_num_tests += 1

            # pred = pred.round()
            correct += pred.argmax(1).eq(y.argmax(1)).sum().item()


    test_loss /= num_batches
    correct /= size
    # print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")
    # print(f"SANITY Accuracy: {100 * sanity_correct / sanity_num_tests:>0.1f}%")

    return test_loss, correct

In [312]:
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loss, train_accuracy = train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loss, test_accuracy = test_loop(test_dataloader, model, loss_fn)

    print(f"Train Error: \n Accuracy: {(100*train_accuracy):>0.1f}%, Avg loss: {train_loss:>8f} \n")
    print(f"Test Error: \n Accuracy: {(100*test_accuracy):>0.1f}%, Avg loss: {test_loss:>8f} \n")
print("Done!")

Epoch 1
-------------------------------
loss: 0.721058  [   64/ 2023]
Train Error: 
 Accuracy: 50.0%, Avg loss: 0.700250 

Test Error: 
 Accuracy: 50.1%, Avg loss: 0.696703 

Epoch 2
-------------------------------
loss: 0.695152  [   64/ 2023]
Train Error: 
 Accuracy: 50.1%, Avg loss: 0.695407 

Test Error: 
 Accuracy: 49.9%, Avg loss: 0.695178 

Epoch 3
-------------------------------
loss: 0.689962  [   64/ 2023]
Train Error: 
 Accuracy: 51.0%, Avg loss: 0.693663 

Test Error: 
 Accuracy: 49.8%, Avg loss: 0.694686 

Epoch 4
-------------------------------
loss: 0.694622  [   64/ 2023]
Train Error: 
 Accuracy: 51.5%, Avg loss: 0.693274 

Test Error: 
 Accuracy: 50.4%, Avg loss: 0.694347 

Epoch 5
-------------------------------
loss: 0.690919  [   64/ 2023]
Train Error: 
 Accuracy: 51.4%, Avg loss: 0.692636 

Test Error: 
 Accuracy: 50.2%, Avg loss: 0.693984 

Epoch 6
-------------------------------
loss: 0.685518  [   64/ 2023]
Train Error: 
 Accuracy: 51.1%, Avg loss: 0.692518 

Te